# Notebook 01 — Stochastic Processes

Demonstrates the AR(1) supply and price processes from §2.2 of Löhndorf & Minner (2009).

**Goals:**
- Verify that simulated paths have the intended mean, variance, autocorrelation, and cross-correlation.
- Visualise sample paths under different parameter configurations.
- Confirm the derivation of $\theta_{PY}$ from the target correlation $\rho$.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from stochastic import StochasticParams, simulate_path

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Default parameters

Paper defaults (Table 1): $\mu_Y = \mu_P = 5$, $\sigma_Y = \sigma_P = 2$, $\theta_Y = \theta_P = 0.5$, $\rho = -0.5$.

In [ ]:
params = StochasticParams(
    mu_Y=5.0, sigma_Y=2.0, theta_Y=0.5,
    mu_P=5.0, sigma_P=2.0, theta_P=0.5,
    rho=-0.5,
)

print(f"theta_PY     = {params.theta_PY:.4f}   (derived from rho={params.rho})")
print(f"mu_eps_Y     = {params.mu_eps_Y:.4f}")
print(f"sigma_eps_Y  = {params.sigma_eps_Y:.4f}")
print(f"mu_eps_P     = {params.mu_eps_P:.4f}")
print(f"sigma_eps_P  = {params.sigma_eps_P:.4f}")
print(f"Implied rho  = {params.realized_rho():.4f}  (should match {params.rho})")

## 2. Sample paths

In [ ]:
rng = np.random.default_rng(42)
T = 200
Y, P = simulate_path(params.mu_Y, params.mu_P, T, params, rng)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(Y, color='steelblue', lw=0.8)
axes[0].axhline(params.mu_Y, color='k', ls='--', lw=0.8, label=f'$\\mu_Y={params.mu_Y}$')
axes[0].set_ylabel('Supply $y$')
axes[0].legend(loc='upper right')
axes[0].set_title('AR(1) Supply and Price Paths')

axes[1].plot(P, color='tomato', lw=0.8)
axes[1].axhline(params.mu_P, color='k', ls='--', lw=0.8, label=f'$\\mu_P={params.mu_P}$')
axes[1].set_ylabel('Price $p$')
axes[1].set_xlabel('Period')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

## 3. Moment verification

Simulate a long path and verify sample moments match the targets.

In [ ]:
rng = np.random.default_rng(0)
T_long = 100_000
Y_long, P_long = simulate_path(params.mu_Y, params.mu_P, T_long, params, rng)

# Drop initial transient
burn = 1000
Ys, Ps = Y_long[burn:], P_long[burn:]

def autocorr(x, lag=1):
    return np.corrcoef(x[:-lag], x[lag:])[0, 1]

corr_YP = np.corrcoef(Ys, Ps)[0, 1]

print("Statistic        Target    Simulated")
print(f"E[Y]             {params.mu_Y:.2f}      {Ys.mean():.4f}")
print(f"Std[Y]           {params.sigma_Y:.2f}      {Ys.std():.4f}")
print(f"Autocorr(Y,1)    {params.theta_Y:.2f}      {autocorr(Ys):.4f}")
print(f"E[P]             {params.mu_P:.2f}      {Ps.mean():.4f}")
print(f"Std[P]           {params.sigma_P:.2f}      {Ps.std():.4f}")
print(f"Autocorr(P,1)    {params.theta_P:.2f}      {autocorr(Ps):.4f}")
print(f"Corr(Y,P)        {params.rho:.2f}      {corr_YP:.4f}")

## 4. Sensitivity: effect of autocorrelation and correlation

In [ ]:
configs = [
    ('Default ($\\theta_Y=0.5$, $\\rho=-0.5$)', dict(theta_Y=0.5, theta_P=0.5, rho=-0.5)),
    ('High autocorr ($\\theta_Y=0.75$)', dict(theta_Y=0.75, theta_P=0.75, rho=-0.5)),
    ('No autocorr ($\\theta_Y=0$)', dict(theta_Y=0.0, theta_P=0.0, rho=-0.5)),
    ('Positive correlation ($\\rho=0$)', dict(theta_Y=0.5, theta_P=0.5, rho=0.0)),
]

fig, axes = plt.subplots(len(configs), 2, figsize=(13, 10), sharex=True)
T_plot = 150

for i, (label, kw) in enumerate(configs):
    p = StochasticParams(**{**dict(mu_Y=5, sigma_Y=2, mu_P=5, sigma_P=2), **kw})
    rng = np.random.default_rng(7)
    Y_, P_ = simulate_path(p.mu_Y, p.mu_P, T_plot, p, rng)
    axes[i, 0].plot(Y_, color='steelblue', lw=0.8)
    axes[i, 0].axhline(p.mu_Y, color='k', ls='--', lw=0.6)
    axes[i, 0].set_ylabel(label, fontsize=9)
    if i == 0:
        axes[i, 0].set_title('Supply $y$')
    axes[i, 1].plot(P_, color='tomato', lw=0.8)
    axes[i, 1].axhline(p.mu_P, color='k', ls='--', lw=0.6)
    if i == 0:
        axes[i, 1].set_title('Price $p$')

axes[-1, 0].set_xlabel('Period')
axes[-1, 1].set_xlabel('Period')
plt.suptitle('Sample paths for different parameter configurations', y=1.01)
plt.tight_layout()
plt.show()

## 5. Scatter plot: supply vs price

Shows the negative correlation $\rho = -0.5$ between supply and price.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, rho_val in zip(axes, [-0.5, 0.0]):
    p = StochasticParams(mu_Y=5, sigma_Y=2, theta_Y=0.5,
                         mu_P=5, sigma_P=2, theta_P=0.5, rho=rho_val)
    rng = np.random.default_rng(1)
    Yp, Pp = simulate_path(p.mu_Y, p.mu_P, 5000, p, rng)
    ax.scatter(Yp[200:], Pp[200:], s=3, alpha=0.3, color='steelblue')
    ax.set_xlabel('Supply $y$')
    ax.set_ylabel('Price $p$')
    ax.set_title(f'$\\rho = {rho_val}$  (sample corr = {np.corrcoef(Yp[200:], Pp[200:])[0,1]:.3f})')

plt.tight_layout()
plt.show()